In [1]:
!nvidia-smi

Fri May  1 11:12:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Quadro T1000                 WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8              3W /   50W |     113MiB /   4096MiB |     18%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [35]:
import os  # library to deal with operating system
import requests # library use to dell with GET/POST requestes

#the url of book we test on 
url= r"https://arxiv.org/pdf/2412.19437"

#this path automatic assinge to the same folder we are her and name the book : "Data_file.pdf"
pdf_path=os.path.join(os.getcwd() , "Data_file.pdf")

# Confirming the existing of book and reinstall if not found 
if not os.path.exists(pdf_path):    
    response= requests.get(url) #downloadin is get request 
    if response.status_code == 200:
        with open(pdf_path,"wb") as f:  # more efficient than `for` in c level 
            f.write(response.content)
        print(f"[INFO] the file is downloaded at : {pdf_path}")
else:
    print(f"[INFO] file exist : {pdf_path} ")

[INFO] file exist : e:\Tests\Data_file.pdf 


In [3]:
# import fitz  # PyMuPDF
# pages_and_texts=[]
# doc = fitz.open("Data_file.pdf")
# data_text = []

# for page_num , page in enumerate(doc):
#     text = page.get_text()
#     pages_and_texts.append({
#         "text_of_page" : text,
#         "page_number" : page_num + 1
#     })
# print(pages_and_texts[3]) 
# for text in pages_and_texts:
#     text["text_of_page"] =text["text_of_page"].replace(r"-\n"," ").strip()
# print(pages_and_texts[3] )

In [36]:
import fitz  # PyMuPDF :{ library used to hadnel PDF files }.

# `clean_text` : remove the spaces from begnining and ending of string
def clean_text(text: str):  
    text = text.replace("\n", " ").strip()
    return text


# open opdf and make array of pages with some statistics and use `clean_text`
def open_pdf(file_path:str):
    document=fitz.open(file_path)
    pages_and_texts=[]
    for page_num , page in enumerate(document):
        text = page.get_text()
        cleaned_text = clean_text(text)
        pages_and_texts.append({
            "text_of_page" : cleaned_text,
            "page_count_char" : len(cleaned_text),
            "page_count_word" : len(cleaned_text.split(" ")),
            "page_count_sentences" : len(cleaned_text.split(". ")),
            "tokens_num" : len(cleaned_text) // 4 , # i dont get it but it is what it is 
            "page_number" : page_num + 1
        })
    return pages_and_texts

pages_and_texts=open_pdf("Data_file.pdf")
pages_and_texts[3]

{'text_of_page': '1. Introduction In recent years, Large Language Models (LLMs) have been undergoing rapid iteration and evolution (Anthropic, 2024; Google, 2024; OpenAI, 2024a), progressively diminishing the gap to- wards Artificial General Intelligence (AGI). Beyond closed-source models, open-source models, including DeepSeek series (DeepSeek-AI, 2024a,b,c; Guo et al., 2024), LLaMA series (AI@Meta, 2024a,b; Touvron et al., 2023a,b), Qwen series (Qwen, 2023, 2024a,b), and Mistral series (Jiang et al., 2023; Mistral, 2024), are also making significant strides, endeavoring to close the gap with their closed-source counterparts. To further push the boundaries of open-source model capa- bilities, we scale up our models and introduce DeepSeek-V3, a large Mixture-of-Experts (MoE) model with 671B parameters, of which 37B are activated for each token. With a forward-looking perspective, we consistently strive for strong model performance and economical costs. Therefore, in terms of architectu

In [37]:
import pandas as pd

# use `DataFrame` to display top 10 rows 
df = pd.DataFrame(pages_and_texts)
df.head(10)

,text_of_page,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
0,DeepSeek-V3 Technical Report DeepSeek-AI resea...,1817,238,11,454,1
1,Contents 1 Introduction 4 2 Architecture 6 2.1...,2634,919,766,658,2
2,4.5.3 Batch-Wise Load Balance VS. Sequence-Wis...,1715,572,462,428,3
3,"1. Introduction In recent years, Large Languag...",4248,594,26,1062,4
4,Training Costs Pre-Training Context Extension ...,3287,468,21,821,5
5,verification and reflection patterns of R1 int...,3475,457,23,868,6
6,… Router Input Hidden 𝐮𝐮𝑡𝑡 Output Hidden 𝐡𝐡𝑡𝑡 ...,1557,262,8,389,7
7,where c𝐾𝑉 𝑡 ∈R𝑑𝑐is the compressed latent vecto...,2236,353,9,559,8
8,where 𝑁𝑠and 𝑁𝑟denote the numbers of shared exp...,2901,455,17,725,9
9,Main Model (Next Token Prediction) Embedding L...,2904,451,23,726,10


In [ ]:
# use this `DataFrame` to do some statistics 
df.describe().round(2)

,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
count,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00
std,799.86,178.18,124.63,199.99,15.44
min,699.00,99.00,1.00,174.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00
50%,2992.00,457.00,23.00,748.00,27.00
75%,3281.00,516.00,29.00,820.00,40.00
max,4248.00,919.00,766.00,1062.00,53.00


# Turn text to sentences 

In [7]:
from spacy.lang.en import English

nlp= English()

# add sentencizer to pipeliine

nlp.add_pipe("sentencizer")

#example sentence to test
semtemces = "this my name . hellow mr.ahmed . what is your name ?"

doc = nlp(semtemces)
# list the sentences to show 
list(doc.sents)


[this my name ., hellow mr.ahmed ., what is your name ?]

# Now try the `data` 

In [8]:
for page in pages_and_texts:
    page["sentences"] = list(nlp(page["text_of_page"]).sents)
    page["sentences_num"] =len(page["sentences"])

#make sure tha all sentences is string not spacy type
for page in pages_and_texts:
    page["sentences"]=[str(sentences) for sentences in page["sentences"]]

pages_and_texts[5]

{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [9]:
#read specific sentence
pages_and_texts[5]["sentences"][6]

'Code, Math, and Reasoning: (1) DeepSeek-V3 achieves state-of-the-art performance on math-related benchmarks among all non-long-CoT open-source and closed-source models.'

In [10]:
# see some statistics about this data 
df =pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num
count,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94
std,799.86,178.18,124.63,199.99,15.44,14.44
min,699.00,99.00,1.00,174.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00


In [11]:
test=list(range(1,25))
input_list=[]
slice_size=10
x=[test[i:i+slice_size] for i in range(0,len(test),slice_size)]
x

[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
 [21, 22, 23, 24]]

In [12]:
def chunker (input_list : list[str],slice_size:int)->list[list[str]]:

    return[input_list[i:i+slice_size] for i in range(0,len(input_list),slice_size)]

for page in pages_and_texts:
    page["sentences_chunk"] = chunker(page["sentences"],slice_size)
    page["sentences_chunk_num"] =len( page["sentences_chunk"]) 

pages_and_texts[5]

{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [13]:
df =pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num,sentences_chunk_num
count,53.00,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94,2.89
std,799.86,178.18,124.63,199.99,15.44,14.44,1.40
min,699.00,99.00,1.00,174.00,1.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00,2.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00,3.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00,3.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00,6.00


In [14]:
pages_and_chunk=[]
for page in pages_and_texts:
    for chunk in page["sentences_chunk"]:
        chunk_dict = {}

        # join all sentences in chunk to be one paragraphe
        paragragh = "".join(chunk).replace("  "," ").strip()
        chunk_dict ["paragraphe"] = paragragh
        
        # Calculate statistics
        words = paragragh.split(" ")
        word_count = len(words)
        char_count = len(paragragh)
        tokens_count = char_count // 4  # approximate tokens
        sentence_count = len(chunk)  # number of sentences in chunk
        
        # Store statistics
        chunk_dict["word_count"] = word_count
        chunk_dict["char_count"] = char_count
        chunk_dict["tokens_count"] = tokens_count
        chunk_dict["sentence_count"] = sentence_count
        chunk_dict ["page_number"] = page["page_number"]

        pages_and_chunk.append(chunk_dict)
pages_and_chunk[5] ,len(pages_and_chunk),len(pages_and_texts)

({'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
  'word_count': 298,
  'char_count': 830,
  'tokens_count': 207,
  'sentence_count': 10,
  'page_number': 3},
 153,
 53)

In [15]:
df = pd.DataFrame(pages_and_chunk)
df.describe().round(2)

,word_count,char_count,tokens_count,sentence_count,page_number
count,153.00,153.00,153.00,153.00,153.00
mean,157.03,974.48,243.26,8.29,26.52
std,145.88,584.64,146.14,2.95,13.86
min,1.00,1.00,0.00,1.00,1.00
25%,74.00,572.00,143.00,7.00,15.00
50%,134.00,912.00,228.00,10.00,28.00
75%,191.00,1301.00,325.00,10.00,39.00
max,877.00,2975.00,743.00,10.00,53.00


In [16]:
df[df["word_count"]<4]
pages_and_texts[8]

{'text_of_page': 'where 𝑁𝑠and 𝑁𝑟denote the numbers of shared experts and routed experts, respectively; FFN(𝑠) 𝑖 (·) and FFN(𝑟) 𝑖 (·) denote the 𝑖-th shared expert and the 𝑖-th routed expert, respectively; 𝐾𝑟denotes the number of activated routed experts; 𝑔𝑖,𝑡is the gating value for the 𝑖-th expert; 𝑠𝑖,𝑡is the token-to-expert affinity; e𝑖is the centroid vector of the 𝑖-th routed expert; and Topk(·, 𝐾) denotes the set comprising 𝐾highest scores among the affinity scores calculated for the 𝑡-th token and all routed experts. Slightly different from DeepSeek-V2, DeepSeek-V3 uses the sigmoid function to compute the affinity scores, and applies a normalization among all selected affinity scores to produce the gating values. Auxiliary-Loss-Free Load Balancing. For MoE models, an unbalanced expert load will lead to routing collapse (Shazeer et al., 2017) and diminish computational efficiency in scenarios with expert parallelism. Conventional solutions usually rely on the auxiliary loss (Fedus e

In [17]:
#filter out tokens less than 30
filtered_chunks = [chunk for chunk in pages_and_chunk if chunk["tokens_count"] >= 30]
filtered_chunks[111]

{'paragraphe': 'A study of bfloat16 for deep learning training.arXiv preprint arXiv:1905.12322, 2019.S. Krishna, K. Krishna, A. Mohananey, S. Schwarcz, A. Stambler, S. Upadhyay, and M. Faruqui.Fact, fetch, and reason: A unified evaluation of retrieval-augmented generation.CoRR, abs/2409.12941, 2024.doi: 10.48550/ARXIV.2409.12941.URL https://doi.org/10.485 50/arXiv.2409.12941.T. Kwiatkowski, J. Palomaki, O. Redfield, M. Collins, A. P. Parikh, C. Alberti, D. Epstein, I. Polosukhin, J. Devlin, K. Lee, K. Toutanova, L. Jones, M. Kelcey, M. Chang, A. M. Dai, J. Uszkoreit, Q. Le, and S. Petrov.Natural questions: a benchmark for question answering research.Trans.',
 'word_count': 84,
 'char_count': 648,
 'tokens_count': 162,
 'sentence_count': 10,
 'page_number': 39}

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Mohamed\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


In [19]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.6.0+cu124
True


In [20]:
set = ["my name", "her name", "the naem"]
embeding = model.encode(set)
embeding = dict(zip(set,embeding))

embeding,embeding["my name"].shape

({'my name': array([-1.11268602e-01,  3.23448591e-02, -3.83405834e-02,  2.23898627e-02,
         -4.03140783e-02, -5.62008433e-02,  2.33329967e-01,  3.92086729e-02,
         -1.01079242e-02, -6.38170391e-02,  1.64090227e-02, -3.45564671e-02,
          4.02090251e-02, -7.56014660e-02, -8.34888965e-03,  2.65754387e-02,
         -1.42914923e-02,  6.17231149e-03, -8.46135616e-02, -7.72886351e-02,
         -4.17721868e-02,  6.27691001e-02, -3.23680506e-05, -2.02984773e-02,
         -8.92573744e-02,  6.10294333e-03, -3.12835611e-02,  9.11155194e-02,
         -3.69639173e-02, -1.61543280e-01, -3.80769518e-04,  7.48009467e-03,
          9.98783186e-02,  5.72648980e-02, -4.66816034e-03, -4.17357311e-02,
         -1.26731142e-01, -2.11646669e-02,  5.62063567e-02, -1.07810488e-02,
         -4.70565679e-03, -1.19292334e-01,  2.42646299e-02,  5.37948310e-03,
          7.11644581e-03,  7.88039621e-03,  6.28924463e-04,  2.45067645e-02,
          1.61230303e-02,  9.47958231e-02, -9.59324986e-02, -8.88

In [21]:
print(type(pages_and_chunk[0]))
print(pages_and_chunk[0])

<class 'dict'>
{'paragraphe': 'DeepSeek-V3 Technical Report DeepSeek-AI research@deepseek.com Abstract We present DeepSeek-V3, a strong Mixture-of-Experts (MoE) language model with 671B total parameters with 37B activated for each token.To achieve efficient inference and cost-effective training, DeepSeek-V3 adopts Multi-head Latent Attention (MLA) and DeepSeekMoE architec- tures, which were thoroughly validated in DeepSeek-V2.Furthermore, DeepSeek-V3 pioneers an auxiliary-loss-free strategy for load balancing and sets a multi-token prediction training objective for stronger performance.We pre-train DeepSeek-V3 on 14.8 trillion diverse and high-quality tokens, followed by Supervised Fine-Tuning and Reinforcement Learning stages to fully harness its capabilities.Comprehensive evaluations reveal that DeepSeek-V3 outperforms other open-source models and achieves performance comparable to leading closed-source models.Despite its excellent performance, DeepSeek-V3 requires only 2.788M H800 G

In [22]:
from tqdm import tqdm

for chunk in tqdm(pages_and_chunk):
    chunk["embedding"] = model.encode(chunk["paragraphe"], device="cuda")


100%|██████████| 153/153 [00:01<00:00, 80.02it/s]


In [23]:
import random
random.sample(pages_and_chunk,k=1)

[{'paragraphe': 'Benchmark (Metric) # Shots DeepSeek-V2 Qwen2.5 LLaMA-3.1 DeepSeek-V3 Base 72B Base 405B Base Base Architecture - MoE Dense Dense MoE # Activated Params - 21B 72B 405B 37B # Total Params - 236B 72B 405B 671B English Pile-test (BPB) - 0.606 0.638 0.542 0.548 BBH (EM) 3-shot 78.8 79.8 82.9 87.5 MMLU (EM) 5-shot 78.4 85.0 84.4 87.1 MMLU-Redux (EM) 5-shot 75.6 83.2 81.3 86.2 MMLU-Pro (EM) 5-shot 51.4 58.3 52.8 64.4 DROP (F1) 3-shot 80.4 80.6 86.0 89.0 ARC-Easy (EM) 25-shot 97.6 98.4 98.4 98.9 ARC-Challenge (EM) 25-shot 92.2 94.5 95.3 95.3 HellaSwag (EM) 10-shot 87.1 84.8 89.2 88.9 PIQA (EM) 0-shot 83.9 82.6 85.9 84.7 WinoGrande (EM) 5-shot 86.3 82.3 85.2 84.9 RACE-Middle (EM) 5-shot 73.1 68.1 74.2 67.1 RACE-High (EM) 5-shot 52.6 50.3 56.8 51.3 TriviaQA (EM) 5-shot 80.0 71.9 82.7 82.9 NaturalQuestions (EM) 5-shot 38.6 33.2 41.5 40.0 AGIEval (EM) 0-shot 57.5 75.8 60.6 79.6 Code HumanEval (Pass@1) 0-shot 43.3 53.0 54.9 65.2 MBPP (Pass@1) 3-shot 65.0 72.6 68.4 75.4 LiveCodeBenc

In [24]:
page_and_Embedding_df = pd.DataFrame(pages_and_chunk)
page_and_Embedding_file_name="page_and_Embedding.csv"
page_and_Embedding_path = os.path.join(os.getcwd(),page_and_Embedding_file_name)
page_and_Embedding_df.to_csv(page_and_Embedding_path,index=False)

In [25]:
pages_and_chunk[5]

{'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 'word_count': 298,
 'char_count': 830,
 'tokens_count': 207,
 'sentence_count': 10,
 'page_number': 3,
 'embedding': array([-5.76343201e-02,  1.862702

In [26]:
pages_and_chunk=[]
pages_and_chunk

[]

In [27]:
pages_and_chunk =pd.read_csv(page_and_Embedding_path)
data = pages_and_chunk.to_dict(orient="records")
data[5]

{'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 'word_count': 298,
 'char_count': 830,
 'tokens_count': 207,
 'sentence_count': 10,
 'page_number': 3,
 'embedding': '[-5.76343201e-02  1.86270270e-02

In [28]:
import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [29]:
data[5]['embedding']

'[-5.76343201e-02  1.86270270e-02 -6.96300194e-02 -5.02142822e-03\n -6.43548667e-02  5.39917648e-02 -4.61324938e-02  3.96139584e-02\n -5.72230220e-02  5.44092804e-03 -4.39483412e-02 -6.78680325e-03\n  4.00274545e-02 -1.59041807e-02  7.62046641e-03  4.98709343e-02\n  1.28310859e-01 -3.56771685e-02 -4.98869978e-02 -8.15589055e-02\n  4.34629358e-02 -4.59041335e-02  8.90600234e-02  1.54037038e-02\n -7.79620484e-02  3.06390580e-02 -5.25193140e-02 -6.64414167e-02\n  3.31790745e-02 -6.93640932e-02 -6.77236542e-03  3.17597482e-03\n  5.10903783e-02 -8.24809521e-02 -8.37938562e-02  1.99515168e-02\n -5.22203930e-02 -5.73823862e-02  1.78819150e-02  2.67783534e-02\n  1.22675279e-04 -3.79560106e-02  2.66227871e-03  2.12215502e-02\n  3.83332558e-02 -2.20188908e-02 -5.95768169e-02 -2.69228555e-02\n -4.48380969e-02 -8.05420727e-02 -1.64256301e-02 -6.48153052e-02\n  5.06796781e-03 -2.43174564e-03 -5.40565290e-02 -8.12954269e-03\n  4.19081375e-02 -2.15617232e-02 -1.57968029e-02  3.31748277e-02\n -8.01364

* the data her is not in list shape so try to fix it

In [30]:
embedding = np.fromstring(data[5]['embedding'].strip('[]'), sep=' ')
embedding

array([-5.76343201e-02,  1.86270270e-02, -6.96300194e-02, -5.02142822e-03,
       -6.43548667e-02,  5.39917648e-02, -4.61324938e-02,  3.96139584e-02,
       -5.72230220e-02,  5.44092804e-03, -4.39483412e-02, -6.78680325e-03,
        4.00274545e-02, -1.59041807e-02,  7.62046641e-03,  4.98709343e-02,
        1.28310859e-01, -3.56771685e-02, -4.98869978e-02, -8.15589055e-02,
        4.34629358e-02, -4.59041335e-02,  8.90600234e-02,  1.54037038e-02,
       -7.79620484e-02,  3.06390580e-02, -5.25193140e-02, -6.64414167e-02,
        3.31790745e-02, -6.93640932e-02, -6.77236542e-03,  3.17597482e-03,
        5.10903783e-02, -8.24809521e-02, -8.37938562e-02,  1.99515168e-02,
       -5.22203930e-02, -5.73823862e-02,  1.78819150e-02,  2.67783534e-02,
        1.22675279e-04, -3.79560106e-02,  2.66227871e-03,  2.12215502e-02,
        3.83332558e-02, -2.20188908e-02, -5.95768169e-02, -2.69228555e-02,
       -4.48380969e-02, -8.05420727e-02, -1.64256301e-02, -6.48153052e-02,
        5.06796781e-03, -

In [31]:
for item in data:
    item['embedding'] = np.fromstring(item['embedding'].strip('[]'), sep=' ')
    

In [32]:
embeddings=[item["embedding"] for item in data]

In [33]:
embeddings

[array([-7.36545399e-02, -5.92405871e-02,  5.75507209e-02,  7.15857744e-02,
         1.54401045e-02, -1.63372401e-02, -7.57386908e-02, -7.95276021e-04,
        -2.40842067e-02, -6.18494265e-02, -1.24292083e-01, -1.00587495e-01,
        -2.54352149e-02, -4.82991673e-02,  1.18151819e-02,  2.85590924e-02,
         5.89606278e-02,  6.40023053e-02, -1.25724867e-01, -8.45907629e-02,
         2.48466004e-02,  1.34899504e-02,  5.55628613e-02, -2.93893926e-02,
         2.66429521e-02,  5.07843383e-02, -1.18937413e-03, -1.09014072e-01,
         1.42404707e-02, -6.30986914e-02,  4.47497219e-02, -3.49188931e-02,
         3.93742993e-02,  1.80439316e-02, -4.82038818e-02,  1.15270421e-01,
        -1.40792534e-01, -8.29446465e-02,  3.17647830e-02, -6.87950626e-02,
         2.28418782e-03,  1.05707645e-02, -1.77034885e-02,  6.14538752e-02,
         5.65050840e-02,  5.83483651e-02,  3.36036719e-02, -7.81908482e-02,
         4.88183610e-02, -2.30751541e-02, -6.04354963e-02, -8.41500685e-02,
         4.4